# ML-1M temporal HNSW / small-world search

This is the next step after exact Top-K temporal attention. It loads the **latest best** Walker+Attention checkpoint from Drive, keeps the trained Q/K/V projections fixed, and asks whether HNSW can recover the useful **Top-16 historical states** without scanning the full history.

References are recomputed from the same checkpoint: native dense attention and exact Top-16 attention. Then HNSW sweeps `efSearch = 16, 32, 64`.

Important: this is a **search-quality diagnostic**, not optimized serving code. A tiny per-user/per-head HNSW index is rebuilt during evaluation; in a streaming system it would be maintained incrementally as events arrive. Build time is reported separately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-temporal-swg-search'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
SRC=f'{REPO}/src'; EXP=f'{REPO}/experiments'
sys.path.insert(0,SRC); sys.path.insert(0,EXP)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'bf16',torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print('BRANCH',BRANCH)


## Run in-process

The checkpoint path is the same directory being updated by the currently-running dense-attention experiment, so this uses whichever `best.pt` exists when this cell starts.

In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/experiments/run_ml1m_temporal_hnsw_search.py'
sys.argv=[SCRIPT,
    '--seed','42',
    '--topk','16',
    '--hnsw-m','8',
    '--ef-construction','80',
    '--ef-search','16','32','64',
    '--eval-batch-size','256',
    '--recall-cap','1024']
print('INPROCESS TEMPORAL HNSW START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('INPROCESS TEMPORAL HNSW END',flush=True)


## Recover compact result

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_temporal_hnsw/result.json')
r=json.loads(p.read_text())
print('CHECKPOINT_EPOCH',r['checkpoint_epoch'])
print('DENSE',r['dense']['NDCG@10'])
print('EXACT_TOP16',r['exact_top16']['NDCG@10'])
print('HNSW')
for row in r['hnsw']:
    print({k:row[k] for k in ['efSearch','NDCG@10','delta_vs_exact_top16','mean_exact_top16_recall','mean_distance_computations_per_head_query']})
print('DECISION',r['decision'])
